<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH4_2021Q4_A100_PARALLEL_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES 3.0 — Notebook 01 Production Batch 4
## ClinVar XML Normalizer — 2021-10 → 2021-12


**Google Colab / GitHub-ready**

This is **GES 3.0 Stage 01, Production Batch 4** for the frozen longitudinal interval:

**October 2021 → December 2021**

Batch 3 completed successfully with the two-source parallel architecture, full-source MD5 verification, structural QC, classification-axis separation, and RCV→SCV linkage QC. Batch 4 keeps that validated production design.

### Execution strategy

**For each month: RCV ∥ VCV in parallel → next month**

- October RCV and VCV run concurrently.
- November RCV and VCV run concurrently.
- December RCV and VCV run concurrently.
- Maximum concurrency remains **2**.
- Every source keeps an independent HTTP session, parser state, checksum, Parquet partition, retry path, and completion marker.


### Production acceptance

A source is complete only after:
1. the full compressed stream reaches EOF;
2. NCBI MD5 matches;
3. Parquet row counts reconcile;
4. structural QC passes;
5. classification semantics remain separated;
6. RCV→SCV linkage passes;
7. a verified `release_complete.json` exists.

The Stage-00 frozen study window remains **2021-01 → 2026-08**.


## 2. Install dependencies

In [3]:
!pip -q install lxml pyarrow pandas requests tqdm beautifulsoup4

## 3. Imports and reproducibility metadata

In [4]:
from __future__ import annotations

import concurrent.futures as cf
import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import bs4
import lxml
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from lxml import etree
from tqdm.auto import tqdm

RUN_UTC = datetime.now(timezone.utc).isoformat()

print("GES 3.0 Stage 01 started:", RUN_UTC)
print("Python:", sys.version.split()[0])
print("lxml:", lxml.__version__)
print("pyarrow:", pa.__version__)
print("pandas:", pd.__version__)

def detect_accelerator():
    try:
        p = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        names = [x.strip() for x in p.stdout.splitlines() if x.strip()]
        return names if names else ["NONE"]
    except Exception:
        return ["NONE"]

DETECTED_GPU_NAMES = detect_accelerator()
HOST_CPU_COUNT = os.cpu_count()

print("Detected GPU(s):", DETECTED_GPU_NAMES)
print("Host logical CPU count:", HOST_CPU_COUNT)
if not any("A100" in x.upper() for x in DETECTED_GPU_NAMES):
    print(
        "NOTE: A100 was not detected. The notebook can still run correctly, "
        "but this run will not be an A100-host benchmark."
    )


GES 3.0 Stage 01 started: 2026-08-17T18:21:07.157312+00:00
Python: 3.12.13
lxml: 6.1.1
pyarrow: 18.1.0
pandas: 2.2.2
Detected GPU(s): ['NVIDIA A100-SXM4-80GB']
Host logical CPU count: 12


## 3A. Accelerator/runtime verification

This check is informational. Stage 01 remains scientifically valid without a GPU, but the purpose of Batch 4 is to benchmark the A100-backed runtime plus two-source concurrency.

In [5]:
print("Detected accelerator(s):", DETECTED_GPU_NAMES)
print("Logical CPUs:", HOST_CPU_COUNT)

A100_DETECTED = any("A100" in x.upper() for x in DETECTED_GPU_NAMES)
print("A100 detected:", A100_DETECTED)

if not A100_DETECTED:
    print(
        "WARNING: This is not an A100 benchmark run. "
        "You can still continue, but record that fact when comparing runtime."
    )

Detected accelerator(s): ['NVIDIA A100-SXM4-80GB']
Logical CPUs: 12
A100 detected: True


## 4. Production configuration

This notebook is already configured for **Batch 4: 2021-10 → 2021-12**.

The source stream is complete, so:
- `record_limit = None`
- NCBI MD5 can be verified
- completion markers can be trusted
- the output can feed Stage 02 after all intended Stage 01 batches are finished

For Colab, Google Drive is enabled by default because the normalized Parquet outputs should survive runtime disconnects.

In [6]:
# ============================================================
# GES 3.0 STAGE 01 — PRODUCTION BATCH 4
# ============================================================

RUN_PROFILE = "PRODUCTION_BATCH"

# Frozen production interval for this notebook.
BATCH_START_MONTH = "2021-10"
BATCH_END_MONTH = "2021-12"

# Process both products.
MODELS_TO_PROCESS = ["RCV", "VCV"]

# Performance experiment: process RCV and VCV concurrently within each month.
MAX_PARALLEL_SOURCES = 2
PARALLELIZE_WITHIN_MONTH = True

# Never increase this above 2 without a separate stress test:
# large XML streams + gzip + Parquet + Drive can saturate RAM/network/storage.


# No pilot truncation in production.
PILOT_RECORD_LIMIT_PER_SOURCE = None

# Persistent storage is strongly recommended for full ClinVar sources.
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive/GES3"
LOCAL_ROOT = "/content/GES3"

# Stream compressed XML directly from NCBI.
STREAM_REMOTE_GZIP = True

# Bounded Parquet shards.
PARQUET_ROWS_PER_SHARD = 50_000

# HTTP/retry behavior.
REQUEST_TIMEOUT = 180
USER_AGENT = "GES3-ClinVar-Normalizer/1.2 batch4-parallel production research pipeline"
MAX_SOURCE_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = 15

# Resume behavior.
RESUME_COMPLETED_RELEASES = True
DELETE_INCOMPLETE_PARTITION_BEFORE_RETRY = True

# Artifact integrity.
HASH_PARQUET_SHARDS = True

# ------------------------------------------------------------
# Stage 00 audited freeze invariants
# ------------------------------------------------------------
FROZEN_START_MONTH = "2021-01"
FROZEN_END_MONTH = "2026-08"
EXPECTED_FROZEN_MONTHS = 68
EXPECTED_CANONICAL_ROWS = 136

EXPECTED_FORMAT_COUNTS = {
    ("RCV", "legacy"): 37,
    ("RCV", "current"): 31,
    ("VCV", "legacy"): 37,
    ("VCV", "current"): 31,
}

print("RUN_PROFILE =", RUN_PROFILE)
print("Batch =", BATCH_START_MONTH, "→", BATCH_END_MONTH)
print("Models =", MODELS_TO_PROCESS)
print("Google Drive =", USE_GOOGLE_DRIVE)
print("Parallel sources/month =", MAX_PARALLEL_SOURCES)


RUN_PROFILE = PRODUCTION_BATCH
Batch = 2021-10 → 2021-12
Models = ['RCV', 'VCV']
Google Drive = True
Parallel sources/month = 2


## 5. Mount persistent Google Drive storage

The default production configuration stores outputs under:

`MyDrive/GES3/stage01/`

If you deliberately want ephemeral local storage, change `USE_GOOGLE_DRIVE = False` in the configuration cell.

In [7]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(GOOGLE_DRIVE_ROOT)
else:
    ROOT = Path(LOCAL_ROOT)

STAGE00_DIR = ROOT / "stage00"
STAGE01_DIR = ROOT / "stage01"
DATA_DIR = STAGE01_DIR / "normalized"
QC_DIR = STAGE01_DIR / "qc"
META_DIR = STAGE01_DIR / "metadata"

for d in [STAGE00_DIR, STAGE01_DIR, DATA_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Stage 00 directory:", STAGE00_DIR)
print("Stage 01 output:", STAGE01_DIR)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/GES3
Stage 00 directory: /content/drive/MyDrive/GES3/stage00
Stage 01 output: /content/drive/MyDrive/GES3/stage01


## 6. Load a frozen Stage 00 artifact if available

Preferred inputs:
- `clinvar_release_manifest_canonical.csv`
- `GES3_STAGE00_ARTIFACTS.zip`

The notebook searches common Colab and Google Drive paths automatically.

If neither is found, the next cell performs a **strict Stage 00 lock reconstruction** for the already-audited 2021-01 → 2026-08 window. That fallback is production-safe only because it must reproduce the exact Stage 00 audit counts before parsing is allowed.

In [8]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

def find_stage00_artifact():
    candidates = [
        Path("/content/clinvar_release_manifest_canonical.csv"),
        Path("/content/ges3_stage00/clinvar_release_manifest_canonical.csv"),
        STAGE00_DIR / "clinvar_release_manifest_canonical.csv",
        STAGE00_DIR / "imported_stage00" / "clinvar_release_manifest_canonical.csv",
        Path("/content/GES3_STAGE00_ARTIFACTS.zip"),
        STAGE00_DIR / "GES3_STAGE00_ARTIFACTS.zip",
    ]

    for p in candidates:
        if p.exists():
            return p

    for p in Path("/content").glob("*STAGE00*.zip"):
        return p
    for p in Path("/content").glob("*release_manifest_canonical*.csv"):
        return p

    return None

def load_manifest_from_artifact(path: Path) -> tuple[pd.DataFrame, str]:
    path = Path(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path), f"frozen_csv:{path}"

    if path.suffix.lower() == ".zip":
        extract_dir = STAGE00_DIR / "imported_stage00"
        extract_dir.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(path), str(extract_dir))
        hits = list(extract_dir.rglob("clinvar_release_manifest_canonical.csv"))
        if not hits:
            raise FileNotFoundError(
                "Stage 00 ZIP did not contain clinvar_release_manifest_canonical.csv"
            )
        return pd.read_csv(hits[0]), f"frozen_zip:{path}"

    raise ValueError(f"Unsupported Stage 00 artifact: {path}")

stage00_artifact = find_stage00_artifact()
manifest = None
manifest_origin = None

if stage00_artifact is not None:
    manifest, manifest_origin = load_manifest_from_artifact(stage00_artifact)
    print("Loaded Stage 00 artifact:", stage00_artifact)
else:
    print("No Stage 00 artifact found locally.")
    print("Will attempt strict reconstruction against the audited freeze invariants.")

Loaded Stage 00 artifact: /content/drive/MyDrive/GES3/stage00/clinvar_release_manifest_canonical.csv


## 7. Strict Stage 00 production lock

This cell does **not** create a moving "latest" cohort.

It reconstructs only the already-audited interval:

**2021-01 through 2026-08**

and refuses to proceed unless the canonical inventory exactly reproduces the known Stage 00 structure.

After validation, it writes a local `clinvar_release_manifest_canonical.csv` and SHA-256. That file becomes the frozen source manifest used by this and later production batches.

In [9]:
BASE = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/"

SOURCE_REGISTRY = pd.DataFrame([
    {
        "data_model": "VCV",
        "format_generation": "current",
        "root_url": BASE,
        "archive_url": BASE + "archive/",
        "filename_prefix": "ClinVarVCVRelease_",
    },
    {
        "data_model": "RCV",
        "format_generation": "current",
        "root_url": BASE + "RCV_release/",
        "archive_url": BASE + "RCV_release/archive/",
        "filename_prefix": "ClinVarRCVRelease_",
    },
    {
        "data_model": "VCV",
        "format_generation": "legacy",
        "root_url": BASE + "VCV_xml_old_format/",
        "archive_url": BASE + "VCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarVariationRelease_",
    },
    {
        "data_model": "RCV",
        "format_generation": "legacy",
        "root_url": BASE + "RCV_xml_old_format/",
        "archive_url": BASE + "RCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarFullRelease_",
    },
])

def get_text(url: str) -> str:
    r = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

def list_apache_links(url: str) -> pd.DataFrame:
    soup = BeautifulSoup(get_text(url), "html.parser")
    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href in ("../", "/"):
            continue
        rows.append({
            "name": a.get_text(" ", strip=True) or href,
            "href": href,
            "url": urljoin(url, href),
        })
    return pd.DataFrame(rows)

def discover_archive_years(archive_url: str) -> list[int]:
    try:
        idx = list_apache_links(archive_url)
    except Exception:
        return []
    years = []
    for href in idx.get("href", []):
        m = re.fullmatch(r"(\d{4})/", str(href))
        if m:
            years.append(int(m.group(1)))
    return sorted(set(years))

def parse_release_month(name: str) -> str | None:
    m = re.search(r"_(\d{4})-(\d{2})\.xml\.gz$", str(name))
    return f"{m.group(1)}-{m.group(2)}" if m else None

def fetch_md5(md5_url: str) -> str | None:
    try:
        text = get_text(md5_url)
        m = re.search(r"\b([a-fA-F0-9]{32})\b", text)
        return m.group(1).lower() if m else None
    except Exception:
        return None

def discover_source(row: pd.Series) -> pd.DataFrame:
    prefix = row["filename_prefix"]
    pattern = re.compile(re.escape(prefix) + r"\d{4}-\d{2}\.xml\.gz$")
    frames = []

    try:
        root = list_apache_links(row["root_url"])
        root["scope"] = "root"
        frames.append(root)
    except Exception as e:
        print("Root listing warning:", row["root_url"], repr(e))

    for year in discover_archive_years(row["archive_url"]):
        # Freeze to 2021-2026 only.
        if year < 2021 or year > 2026:
            continue
        u = urljoin(row["archive_url"], f"{year}/")
        try:
            d = list_apache_links(u)
            d["scope"] = f"archive/{year}"
            frames.append(d)
        except Exception as e:
            print("Archive listing warning:", u, repr(e))

    if not frames:
        return pd.DataFrame()

    x = pd.concat(frames, ignore_index=True)
    x = x[x["name"].map(lambda n: bool(pattern.fullmatch(str(n))))].copy()
    x["data_model"] = row["data_model"]
    x["format_generation"] = row["format_generation"]
    x["release_month"] = x["name"].map(parse_release_month)
    x["md5_url"] = x["url"] + ".md5"

    # Hard freeze the previously audited interval.
    p = pd.PeriodIndex(x["release_month"], freq="M")
    keep = (
        (p >= pd.Period(FROZEN_START_MONTH, freq="M"))
        & (p <= pd.Period(FROZEN_END_MONTH, freq="M"))
    )
    return x[keep].copy()

def reconstruct_and_lock_stage00_manifest():
    parts = []

    for _, row in SOURCE_REGISTRY.iterrows():
        print("Discovering:", row["data_model"], row["format_generation"])
        part = discover_source(row)
        if not part.empty:
            parts.append(part)

    if not parts:
        raise RuntimeError("Could not reconstruct the frozen ClinVar inventory.")

    all_files = pd.concat(parts, ignore_index=True).drop_duplicates(
        ["data_model", "format_generation", "release_month", "url"]
    )

    all_files["priority"] = all_files["format_generation"].map(
        {"current": 0, "legacy": 1}
    )

    canonical = (
        all_files
        .sort_values(["data_model", "release_month", "priority", "url"])
        .groupby(["data_model", "release_month"], as_index=False)
        .head(1)
        .sort_values(["release_month", "data_model"])
        .reset_index(drop=True)
    )

    expected_months = pd.period_range(
        FROZEN_START_MONTH, FROZEN_END_MONTH, freq="M"
    ).astype(str)

    # Structural freeze checks.
    assert len(expected_months) == EXPECTED_FROZEN_MONTHS
    assert len(canonical) == EXPECTED_CANONICAL_ROWS, (
        f"Stage 00 lock mismatch: expected {EXPECTED_CANONICAL_ROWS} canonical rows, "
        f"found {len(canonical)}"
    )

    for model in ["RCV", "VCV"]:
        observed_months = set(
            canonical.loc[canonical["data_model"] == model, "release_month"]
        )
        missing = sorted(set(expected_months) - observed_months)
        extra = sorted(observed_months - set(expected_months))
        assert not missing and not extra, (
            f"{model} frozen month mismatch. Missing={missing}, extra={extra}"
        )

    counts = (
        canonical.groupby(["data_model", "format_generation"])
        .size()
        .to_dict()
    )
    for key, expected_n in EXPECTED_FORMAT_COUNTS.items():
        observed_n = int(counts.get(key, 0))
        assert observed_n == expected_n, (
            f"Stage 00 format-count mismatch for {key}: "
            f"expected {expected_n}, got {observed_n}"
        )

    # Fetch NCBI MD5 for every canonical row once, then freeze it locally.
    md5_values = []
    for _, r in tqdm(canonical.iterrows(), total=len(canonical), desc="Freezing NCBI MD5"):
        md5_values.append(fetch_md5(r["md5_url"]))

    canonical["ncbi_md5"] = md5_values

    missing_md5 = canonical["ncbi_md5"].isna().sum()
    if missing_md5:
        raise RuntimeError(
            f"Could not freeze NCBI MD5 for {missing_md5} canonical sources."
        )

    locked_path = STAGE00_DIR / "clinvar_release_manifest_canonical.csv"
    canonical.to_csv(locked_path, index=False)

    locked_sha = hashlib.sha256(locked_path.read_bytes()).hexdigest()
    (STAGE00_DIR / "clinvar_release_manifest_canonical.sha256").write_text(
        locked_sha + "  " + locked_path.name + "\n",
        encoding="utf-8",
    )

    audit = {
        "frozen_start_month": FROZEN_START_MONTH,
        "frozen_end_month": FROZEN_END_MONTH,
        "expected_months": EXPECTED_FROZEN_MONTHS,
        "canonical_rows": len(canonical),
        "format_counts": {
            f"{k[0]}_{k[1]}": int(v) for k, v in counts.items()
        },
        "manifest_sha256": locked_sha,
        "locked_utc": datetime.now(timezone.utc).isoformat(),
    }

    (STAGE00_DIR / "stage00_production_lock.json").write_text(
        json.dumps(audit, indent=2),
        encoding="utf-8",
    )

    return canonical, f"strict_stage00_reconstruction:{locked_path}", locked_sha

if manifest is None:
    manifest, manifest_origin, manifest_sha256 = reconstruct_and_lock_stage00_manifest()
else:
    # Restrict even an imported artifact to the audited frozen interval.
    manifest["release_month"] = manifest["release_month"].astype(str)
    p = pd.PeriodIndex(manifest["release_month"], freq="M")
    manifest = manifest[
        (p >= pd.Period(FROZEN_START_MONTH, freq="M"))
        & (p <= pd.Period(FROZEN_END_MONTH, freq="M"))
    ].copy()

    manifest["data_model"] = manifest["data_model"].astype(str).str.upper()
    manifest["format_generation"] = manifest["format_generation"].astype(str).str.lower()

    # Exact freeze invariants must still hold.
    assert len(manifest) == EXPECTED_CANONICAL_ROWS, (
        f"Imported Stage 00 manifest has {len(manifest)} rows; "
        f"expected {EXPECTED_CANONICAL_ROWS}."
    )

    counts = manifest.groupby(["data_model", "format_generation"]).size().to_dict()
    for key, expected_n in EXPECTED_FORMAT_COUNTS.items():
        assert int(counts.get(key, 0)) == expected_n, (
            f"Imported Stage 00 manifest mismatch for {key}: "
            f"expected {expected_n}, got {counts.get(key, 0)}"
        )

    frozen_copy = STAGE00_DIR / "clinvar_release_manifest_canonical.csv"
    manifest.sort_values(["release_month", "data_model"]).to_csv(
        frozen_copy, index=False
    )
    manifest_sha256 = hashlib.sha256(frozen_copy.read_bytes()).hexdigest()

required_manifest_cols = {
    "release_month", "data_model", "format_generation", "url", "ncbi_md5"
}
missing_cols = required_manifest_cols - set(manifest.columns)
if missing_cols:
    raise ValueError(f"Frozen manifest missing columns: {sorted(missing_cols)}")

manifest = manifest.sort_values(
    ["release_month", "data_model"]
).reset_index(drop=True)

print("\nSTAGE 00 PRODUCTION LOCK PASSED")
print("Manifest origin:", manifest_origin)
print("Frozen manifest SHA-256:", manifest_sha256)
print("Rows:", len(manifest))
display(
    manifest.groupby(["data_model", "format_generation"])
    .size().rename("n").reset_index()
)


STAGE 00 PRODUCTION LOCK PASSED
Manifest origin: frozen_csv:/content/drive/MyDrive/GES3/stage00/clinvar_release_manifest_canonical.csv
Frozen manifest SHA-256: 8b483f722e7271d51fe9b343df3a385804f2ff48c54701a13311f6c87b246c5f
Rows: 136


,data_model,format_generation,n
0,RCV,current,31
1,RCV,legacy,37
2,VCV,current,31
3,VCV,legacy,37


## 8. Freeze Batch 4 source list

This production notebook selects exactly:

**2021-01, 2021-02, 2021-03 × RCV/VCV = 6 complete source streams**

The batch run manifest is written **before parsing** and receives its own SHA-256.

In [10]:
def month_between(s: pd.Series, start: str, end: str) -> pd.Series:
    p = pd.PeriodIndex(s.astype(str), freq="M")
    return (
        (p >= pd.Period(start, freq="M"))
        & (p <= pd.Period(end, freq="M"))
    )

selected = manifest[
    month_between(
        manifest["release_month"],
        BATCH_START_MONTH,
        BATCH_END_MONTH,
    )
    & manifest["data_model"].isin(MODELS_TO_PROCESS)
].copy()

selected = selected.sort_values(
    ["release_month", "data_model"]
).reset_index(drop=True)

record_limit = None

EXPECTED_BATCH_MONTHS = 3
EXPECTED_BATCH_SOURCES = EXPECTED_BATCH_MONTHS * len(MODELS_TO_PROCESS)

assert len(selected) == EXPECTED_BATCH_SOURCES, (
    f"Expected {EXPECTED_BATCH_SOURCES} source files, found {len(selected)}"
)

for month in pd.period_range(
    BATCH_START_MONTH, BATCH_END_MONTH, freq="M"
).astype(str):
    observed_models = set(
        selected.loc[selected["release_month"] == month, "data_model"]
    )
    assert observed_models == set(MODELS_TO_PROCESS), (
        f"{month}: expected {MODELS_TO_PROCESS}, got {sorted(observed_models)}"
    )

run_manifest_path = META_DIR / "stage01_batch4_2021Q4_run_manifest.csv"
selected.to_csv(run_manifest_path, index=False)

run_manifest_sha256 = hashlib.sha256(
    run_manifest_path.read_bytes()
).hexdigest()

(META_DIR / "stage01_batch4_2021Q4_run_manifest.sha256").write_text(
    run_manifest_sha256 + "  " + run_manifest_path.name + "\n",
    encoding="utf-8",
)

print("Batch source files:", len(selected))
print("Record limit:", record_limit)
print("Batch manifest SHA-256:", run_manifest_sha256)

display(selected[[
    "release_month",
    "data_model",
    "format_generation",
    "url",
    "ncbi_md5",
]])

Batch source files: 6
Record limit: None
Batch manifest SHA-256: 1d9aa38ce60c5e98fcd8e0051ff7b5e8905fad9994d819170779ec58b095b9e6


,release_month,data_model,format_generation,url,ncbi_md5
0,2021-10,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,c67eed9a99a1c33ce335c00651a0e6af
1,2021-10,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,ef02a76001cea1e6fb54ecaadc075ce4
2,2021-11,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,f3a84e95f61747a702eb87fc3912a3ff
3,2021-11,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,f9f0e6b48cea62c05cbc2923629c6163
4,2021-12,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,b14249d9184a22989a2b3c4e17bd04e3
5,2021-12,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,ea0a3f142e0f7967e43d7e1a0cccb689


## 9. Normalized table contract

Stage 01 deliberately writes a **stable, explicit schema** rather than arbitrary flattened XML.

### `vcv_state`
One VCV aggregate state per release month.

### `rcv_state`
One RCV variant-condition aggregate state per release month. This becomes the primary GES 3.0 prediction-unit foundation.

### `scv_state`
One submitted assertion attached to its parent RCV or VCV record for that release. RCV-derived SCVs are especially important because they retain the variant-condition context.

### `vcv_rcv_link`
Links a VCV aggregate record to RCV accessions visible in that release.

JSON strings are used only for repeated provenance fields (genes, conditions, citations, RCV lists). Later notebooks can explode these into graph tables.

In [11]:
GERMLINE_TERMS = {
    "benign",
    "likely benign",
    "uncertain significance",
    "vus-high",
    "vus-mid",
    "vus-low",
    "likely pathogenic",
    "pathogenic",
    "likely pathogenic, low penetrance",
    "pathogenic, low penetrance",
    "uncertain risk allele",
    "likely risk allele",
    "established risk allele",
    "drug response",
    "association",
    "protective",
    "affects",
    "conflicting data from submitters",
    "conflicting classifications of pathogenicity",
    "other",
    "not provided",
}

VCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "vcv_accession", "vcv_version", "variation_id",
    "variation_type", "variation_name",
    "date_created", "date_last_updated",
    "germline_description", "germline_review_status",
    "germline_date_last_evaluated",
    "somatic_clinical_impact_description",
    "somatic_clinical_impact_review_status",
    "oncogenicity_description", "oncogenicity_review_status",
    "legacy_classification_description",
    "legacy_review_status", "legacy_date_last_evaluated",
    "legacy_germline_candidate",
    "gene_symbols_json", "rcv_accessions_json",
    "citation_ids_json", "scv_count",
]

RCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "rcv_accession", "rcv_version",
    "vcv_accession", "vcv_version",
    "variation_id", "variation_name",
    "germline_description", "germline_review_status",
    "germline_date_last_evaluated",
    "somatic_clinical_impact_description",
    "somatic_clinical_impact_review_status",
    "oncogenicity_description", "oncogenicity_review_status",
    "legacy_classification_description",
    "legacy_review_status", "legacy_date_last_evaluated",
    "legacy_germline_candidate",
    "condition_names_json", "condition_ids_json",
    "gene_symbols_json", "citation_ids_json",
    "scv_count",
]

SCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "source_data_model",
    "parent_vcv_accession", "parent_rcv_accession",
    "scv_accession", "scv_version",
    "classification_type_raw",
    "classification_description",
    "review_status", "date_last_evaluated",
    "submitter_name", "submitter_org_id",
    "submitter_local_key", "submitter_date",
    "assertion_method",
    "collection_method", "origin",
    "condition_names_json", "condition_ids_json",
    "citation_ids_json",
    "germline_term_candidate",
]

VCV_RCV_LINK_COLUMNS = [
    "release_month", "source_url", "source_format",
    "vcv_accession", "vcv_version",
    "rcv_accession", "rcv_version",
]

TABLE_COLUMNS = {
    "vcv_state": VCV_COLUMNS,
    "rcv_state": RCV_COLUMNS,
    "scv_state": SCV_COLUMNS,
    "vcv_rcv_link": VCV_RCV_LINK_COLUMNS,
}

print("Normalized schema contract loaded.")
for name, cols in TABLE_COLUMNS.items():
    print(name, "columns:", len(cols))

Normalized schema contract loaded.
vcv_state columns: 25
rcv_state columns: 25
scv_state columns: 23
vcv_rcv_link columns: 7


## 10. XML utility functions

These functions are namespace-tolerant and intentionally conservative. They look for known ClinVar semantic elements while retaining raw text when exact historical representation differs.

In [12]:
def localname(tag) -> str:
    if not isinstance(tag, str):
        return ""
    if "}" in tag:
        return tag.rsplit("}", 1)[-1]
    return tag

def clean_text(x):
    if x is None:
        return None
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x if x else None

def iter_desc(elem, names=None):
    names = set(names) if names else None
    for x in elem.iter():
        n = localname(x.tag)
        if names is None or n in names:
            yield x

def first_desc(elem, names):
    names = set(names)
    for x in elem.iter():
        if localname(x.tag) in names:
            return x
    return None

def first_text(elem, names):
    x = first_desc(elem, names)
    return clean_text(x.text) if x is not None else None

def attr_any(elem, names):
    if elem is None:
        return None
    for k in names:
        if k in elem.attrib:
            return clean_text(elem.attrib.get(k))
    # Case-insensitive fallback.
    lower = {str(k).lower(): v for k, v in elem.attrib.items()}
    for k in names:
        if str(k).lower() in lower:
            return clean_text(lower[str(k).lower()])
    return None

def child_text(container, preferred_names):
    if container is None:
        return None
    for x in container.iter():
        if localname(x.tag) in preferred_names:
            t = clean_text(x.text)
            if t:
                return t
    return None

def json_list(values):
    vals = []
    seen = set()
    for v in values:
        v = clean_text(v)
        if v and v not in seen:
            vals.append(v)
            seen.add(v)
    return json.dumps(vals, ensure_ascii=False)

def extract_element_values(elem, preferred_type=None):
    out = []
    for x in iter_desc(elem, {"ElementValue"}):
        typ = attr_any(x, ["Type"])
        if preferred_type is None or (
            typ and typ.lower() == preferred_type.lower()
        ):
            t = clean_text(x.text)
            if t:
                out.append(t)
    return out

def extract_citations(elem):
    values = []
    for cit in iter_desc(elem, {"Citation"}):
        for x in cit.iter():
            n = localname(x.tag)
            if n in {"ID", "CitationText", "URL"}:
                t = clean_text(x.text)
                if not t:
                    continue
                src = attr_any(x, ["Source", "Type"])
                values.append(f"{src}:{t}" if src else t)
    return values

def extract_conditions(elem):
    names = []
    ids = []

    for trait in iter_desc(elem, {"Trait"}):
        preferred = []
        fallback = []

        for name_el in trait.iter():
            if localname(name_el.tag) != "Name":
                continue
            vals = extract_element_values(name_el, preferred_type="Preferred")
            preferred.extend(vals)
            if not vals:
                fallback.extend(extract_element_values(name_el))

        names.extend(preferred or fallback)

        for xref in iter_desc(trait, {"XRef"}):
            db = attr_any(xref, ["DB"])
            xid = attr_any(xref, ["ID"])
            if db and xid:
                ids.append(f"{db}:{xid}")

    return names, ids

def extract_gene_symbols(elem):
    symbols = []

    for gene in iter_desc(elem, {"Gene"}):
        for sym in iter_desc(gene, {"Symbol"}):
            vals = extract_element_values(sym, preferred_type="Preferred")
            if not vals:
                vals = extract_element_values(sym)
            symbols.extend(vals)

        # Some schema generations expose Symbol as an attribute.
        s = attr_any(gene, ["Symbol"])
        if s:
            symbols.append(s)

    return symbols

def classification_fields(container):
    if container is None:
        return {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }

    description = child_text(
        container,
        {"Description", "ClinicalSignificanceDescription"},
    )
    review_status = child_text(container, {"ReviewStatus"})
    date_last_evaluated = (
        attr_any(container, ["DateLastEvaluated"])
        or child_text(container, {"DateLastEvaluated"})
    )

    return {
        "description": description,
        "review_status": review_status,
        "date_last_evaluated": date_last_evaluated,
    }

def find_named_container(elem, name):
    for x in elem.iter():
        if localname(x.tag) == name:
            return x
    return None

def current_aggregate_classifications(elem):
    germline = classification_fields(
        find_named_container(elem, "GermlineClassification")
    )
    somatic = classification_fields(
        find_named_container(elem, "SomaticClinicalImpact")
    )
    oncogenic = classification_fields(
        find_named_container(elem, "OncogenicityClassification")
    )

    return germline, somatic, oncogenic

def legacy_aggregate_classification(elem):
    # RCV legacy commonly uses ClinicalSignificance.
    cs = find_named_container(elem, "ClinicalSignificance")
    if cs is not None:
        return classification_fields(cs)

    # VCV legacy aggregate data are represented within InterpretedRecord.
    ir = find_named_container(elem, "InterpretedRecord")
    if ir is not None:
        # Search a narrower interpretation/classification container first.
        for name in ["Interpretation", "Classification"]:
            c = find_named_container(ir, name)
            if c is not None:
                f = classification_fields(c)
                if any(f.values()):
                    return f
        return classification_fields(ir)

    return {
        "description": None,
        "review_status": None,
        "date_last_evaluated": None,
    }

def is_germline_candidate(description):
    if not description:
        return False
    d = clean_text(description).lower()
    if d in GERMLINE_TERMS:
        return True
    # Aggregate conflict strings can include extra wording.
    if "conflicting classifications of pathogenicity" in d:
        return True
    return False

def clear_element(elem):
    elem.clear()
    parent = elem.getparent()
    if parent is not None:
        while elem.getprevious() is not None:
            del parent[0]

print("XML utilities loaded.")

XML utilities loaded.


## 11. Parse VCV aggregate records

The VCV accession is taken from `VariationArchive/@Accession`, consistent with NCBI's identifier documentation. RCV links are taken from `RCVList/RCVAccession`.

Current-format aggregate classification axes are retained separately. Legacy single-classification fields remain explicitly marked as legacy.

In [13]:
def parse_vcv_record(elem, release_month, source_url, source_format):
    vcv_accession = attr_any(elem, ["Accession"])
    if not (vcv_accession and vcv_accession.startswith("VCV")):
        return None, [], []

    vcv_version = attr_any(elem, ["Version"])
    variation_id = attr_any(elem, ["VariationID", "VariationId", "ID"])
    variation_type = attr_any(elem, ["VariationType", "Type"])

    variation_name = (
        attr_any(elem, ["VariationName"])
        or first_text(elem, {"VariationName"})
        or first_text(elem, {"Name"})
    )

    date_created = attr_any(elem, ["DateCreated"])
    date_last_updated = attr_any(
        elem, ["DateLastUpdated", "DateLastUpdate"]
    )

    if source_format == "current":
        germline, somatic, oncogenic = current_aggregate_classifications(elem)
        legacy = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
    else:
        germline = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
        somatic = dict(germline)
        oncogenic = dict(germline)
        legacy = legacy_aggregate_classification(elem)

    genes = extract_gene_symbols(elem)
    citations = extract_citations(elem)

    rcv_links = []
    rcv_accessions = []

    for x in iter_desc(elem, {"RCVAccession"}):
        acc = attr_any(x, ["Accession", "Acc"])
        ver = attr_any(x, ["Version"])
        if acc and acc.startswith("RCV"):
            rcv_accessions.append(acc)
            rcv_links.append({
                "release_month": release_month,
                "source_url": source_url,
                "source_format": source_format,
                "vcv_accession": vcv_accession,
                "vcv_version": vcv_version,
                "rcv_accession": acc,
                "rcv_version": ver,
            })

    scvs = parse_scv_children(
        elem=elem,
        source_data_model="VCV",
        parent_vcv_accession=vcv_accession,
        parent_rcv_accession=None,
        release_month=release_month,
        source_url=source_url,
        source_format=source_format,
    )

    row = {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "vcv_accession": vcv_accession,
        "vcv_version": vcv_version,
        "variation_id": variation_id,
        "variation_type": variation_type,
        "variation_name": variation_name,
        "date_created": date_created,
        "date_last_updated": date_last_updated,

        "germline_description": germline["description"],
        "germline_review_status": germline["review_status"],
        "germline_date_last_evaluated": germline["date_last_evaluated"],

        "somatic_clinical_impact_description": somatic["description"],
        "somatic_clinical_impact_review_status": somatic["review_status"],

        "oncogenicity_description": oncogenic["description"],
        "oncogenicity_review_status": oncogenic["review_status"],

        "legacy_classification_description": legacy["description"],
        "legacy_review_status": legacy["review_status"],
        "legacy_date_last_evaluated": legacy["date_last_evaluated"],
        "legacy_germline_candidate": is_germline_candidate(
            legacy["description"]
        ),

        "gene_symbols_json": json_list(genes),
        "rcv_accessions_json": json_list(rcv_accessions),
        "citation_ids_json": json_list(citations),
        "scv_count": len(scvs),
    }

    return row, rcv_links, scvs

## 12. Parse SCV submitted assertions

NCBI documents different SCV accession attribute names across the VCV and RCV XML products:
- VCV: `ClinicalAssertion/ClinVarAccession/@Accession`
- RCV: `ClinVarAssertion/ClinVarAccession/@Acc`

The parser handles both while preserving the source data model.

In [14]:
def infer_submission_meta(assertion):
    submitter_name = None
    submitter_org_id = None
    submitter_local_key = None
    submitter_date = None

    # ClinVarSubmissionID is present in multiple schema generations.
    sid = find_named_container(assertion, "ClinVarSubmissionID")
    if sid is not None:
        submitter_name = attr_any(
            sid, ["submitter", "Submitter", "SubmitterName"]
        )
        submitter_org_id = attr_any(
            sid, ["OrgID", "orgID", "SubmitterID"]
        )
        submitter_local_key = attr_any(
            sid, ["localKey", "LocalKey", "submitterKey"]
        )
        submitter_date = attr_any(
            sid, ["submitterDate", "SubmitterDate", "Date"]
        )

    # Fallbacks for newer representations.
    if submitter_name is None:
        for x in assertion.iter():
            n = localname(x.tag)
            if n in {"Submitter", "Organization"}:
                submitter_name = (
                    attr_any(x, ["Name", "name", "SubmitterName"])
                    or clean_text(x.text)
                )
                submitter_org_id = submitter_org_id or attr_any(
                    x, ["OrgID", "ID", "SubmitterID"]
                )
                if submitter_name:
                    break

    return (
        submitter_name,
        submitter_org_id,
        submitter_local_key,
        submitter_date,
    )

def infer_assertion_classification(assertion, source_format):
    # Current VCV/RCV SCVs use Classification.
    c = find_named_container(assertion, "Classification")
    if c is not None:
        f = classification_fields(c)
        ctype = attr_any(c, ["Type", "ClassificationType"])
        return ctype, f

    # Legacy RCV commonly uses ClinicalSignificance.
    c = find_named_container(assertion, "ClinicalSignificance")
    if c is not None:
        return "legacy_single", classification_fields(c)

    # Legacy VCV may expose interpretation data under the assertion.
    c = find_named_container(assertion, "Interpretation")
    if c is not None:
        return "legacy_single", classification_fields(c)

    # Last resort: keep review/date if directly nested.
    f = classification_fields(assertion)
    return (
        "legacy_single" if source_format == "legacy" else None,
        f,
    )

def extract_assertion_method(assertion):
    # Prefer explicit method text.
    for name in [
        "AssertionMethod",
        "Method",
        "Description",
    ]:
        x = find_named_container(assertion, name)
        if x is not None:
            t = clean_text(x.text)
            if t:
                return t
    return None

def parse_one_scv(
    assertion,
    source_data_model,
    parent_vcv_accession,
    parent_rcv_accession,
    release_month,
    source_url,
    source_format,
):
    acc_el = find_named_container(assertion, "ClinVarAccession")
    if acc_el is None:
        return None

    scv_accession = attr_any(acc_el, ["Accession", "Acc"])
    if not (scv_accession and scv_accession.startswith("SCV")):
        return None

    scv_version = attr_any(acc_el, ["Version"])
    ctype, cf = infer_assertion_classification(assertion, source_format)

    (
        submitter_name,
        submitter_org_id,
        submitter_local_key,
        submitter_date,
    ) = infer_submission_meta(assertion)

    condition_names, condition_ids = extract_conditions(assertion)
    citations = extract_citations(assertion)

    collection_method = first_text(
        assertion,
        {"MethodType", "CollectionMethod"},
    )
    origin = first_text(
        assertion,
        {"Origin", "AlleleOrigin"},
    )
    assertion_method = extract_assertion_method(assertion)

    description = cf["description"]

    return {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "source_data_model": source_data_model,
        "parent_vcv_accession": parent_vcv_accession,
        "parent_rcv_accession": parent_rcv_accession,
        "scv_accession": scv_accession,
        "scv_version": scv_version,
        "classification_type_raw": ctype,
        "classification_description": description,
        "review_status": cf["review_status"],
        "date_last_evaluated": cf["date_last_evaluated"],
        "submitter_name": submitter_name,
        "submitter_org_id": submitter_org_id,
        "submitter_local_key": submitter_local_key,
        "submitter_date": submitter_date,
        "assertion_method": assertion_method,
        "collection_method": collection_method,
        "origin": origin,
        "condition_names_json": json_list(condition_names),
        "condition_ids_json": json_list(condition_ids),
        "citation_ids_json": json_list(citations),
        "germline_term_candidate": is_germline_candidate(description),
    }

def parse_scv_children(
    elem,
    source_data_model,
    parent_vcv_accession,
    parent_rcv_accession,
    release_month,
    source_url,
    source_format,
):
    assertion_tag = (
        "ClinicalAssertion" if source_data_model == "VCV"
        else "ClinVarAssertion"
    )

    rows = []
    for assertion in iter_desc(elem, {assertion_tag}):
        row = parse_one_scv(
            assertion=assertion,
            source_data_model=source_data_model,
            parent_vcv_accession=parent_vcv_accession,
            parent_rcv_accession=parent_rcv_accession,
            release_month=release_month,
            source_url=source_url,
            source_format=source_format,
        )
        if row is not None:
            rows.append(row)

    return rows

print("SCV parser loaded.")

SCV parser loaded.


## 13. Parse RCV aggregate records

The RCV accession comes from `ReferenceClinVarAssertion/ClinVarAccession`, while VCV identity can be represented through `MeasureSet/@Acc` and Variation ID through `MeasureSet/@ID`, consistent with NCBI's accession documentation.

The RCV-level condition and SCV relationship are preserved because the primary future GES 3.0 prediction unit is the **RCV / variant-condition monthly state**.

In [15]:
def parse_rcv_record(elem, release_month, source_url, source_format):
    ref = find_named_container(elem, "ReferenceClinVarAssertion")
    if ref is None:
        return None, []

    acc_el = find_named_container(ref, "ClinVarAccession")
    rcv_accession = attr_any(acc_el, ["Acc", "Accession"])
    if not (rcv_accession and rcv_accession.startswith("RCV")):
        return None, []

    rcv_version = attr_any(acc_el, ["Version"])

    measure_set = find_named_container(ref, "MeasureSet")
    variation_id = attr_any(measure_set, ["ID", "VariationID"])
    vcv_accession = attr_any(measure_set, ["Acc", "Accession"])
    vcv_version = attr_any(measure_set, ["Version"])

    variation_name = (
        first_text(measure_set, {"Name", "ElementValue"})
        if measure_set is not None
        else None
    )

    if source_format == "current":
        germline, somatic, oncogenic = current_aggregate_classifications(ref)
        legacy = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
    else:
        germline = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
        somatic = dict(germline)
        oncogenic = dict(germline)
        legacy = legacy_aggregate_classification(ref)

    condition_names, condition_ids = extract_conditions(ref)
    genes = extract_gene_symbols(ref)
    citations = extract_citations(ref)

    scvs = parse_scv_children(
        elem=elem,
        source_data_model="RCV",
        parent_vcv_accession=vcv_accession,
        parent_rcv_accession=rcv_accession,
        release_month=release_month,
        source_url=source_url,
        source_format=source_format,
    )

    row = {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "rcv_accession": rcv_accession,
        "rcv_version": rcv_version,
        "vcv_accession": vcv_accession,
        "vcv_version": vcv_version,
        "variation_id": variation_id,
        "variation_name": variation_name,

        "germline_description": germline["description"],
        "germline_review_status": germline["review_status"],
        "germline_date_last_evaluated": germline["date_last_evaluated"],

        "somatic_clinical_impact_description": somatic["description"],
        "somatic_clinical_impact_review_status": somatic["review_status"],

        "oncogenicity_description": oncogenic["description"],
        "oncogenicity_review_status": oncogenic["review_status"],

        "legacy_classification_description": legacy["description"],
        "legacy_review_status": legacy["review_status"],
        "legacy_date_last_evaluated": legacy["date_last_evaluated"],
        "legacy_germline_candidate": is_germline_candidate(
            legacy["description"]
        ),

        "condition_names_json": json_list(condition_names),
        "condition_ids_json": json_list(condition_ids),
        "gene_symbols_json": json_list(genes),
        "citation_ids_json": json_list(citations),
        "scv_count": len(scvs),
    }

    return row, scvs

## 14. Parquet shard writer

Rows are written in bounded batches. No release needs to reside fully in RAM.

In [16]:
class ShardWriter:
    def __init__(self, release_dir: Path, table_name: str, columns: list[str]):
        self.dir = release_dir / table_name
        self.dir.mkdir(parents=True, exist_ok=True)
        self.table_name = table_name
        self.columns = columns
        self.buffer = []
        self.part = 0
        self.total_rows = 0

    def add(self, row):
        if row is None:
            return
        normalized = {c: row.get(c) for c in self.columns}
        self.buffer.append(normalized)
        if len(self.buffer) >= PARQUET_ROWS_PER_SHARD:
            self.flush()

    def add_many(self, rows):
        for row in rows:
            self.add(row)

    def flush(self):
        if not self.buffer:
            return

        df = pd.DataFrame(self.buffer, columns=self.columns)

        # Keep string-like schema stable while preserving boolean/numeric fields.
        table = pa.Table.from_pandas(df, preserve_index=False)
        path = self.dir / f"part-{self.part:05d}.parquet"
        pq.write_table(
            table,
            path,
            compression="zstd",
            use_dictionary=True,
        )

        self.total_rows += len(self.buffer)
        self.part += 1
        self.buffer.clear()

    def close(self):
        self.flush()

def make_writers(release_dir: Path):
    return {
        name: ShardWriter(release_dir, name, cols)
        for name, cols in TABLE_COLUMNS.items()
    }

print("Parquet shard writer loaded.")

Parquet shard writer loaded.


## 15. Streaming compressed-byte hashing

For a **full** source parse, the compressed-byte MD5 is calculated while NCBI's gzip stream is being consumed. At end of stream it is compared with the Stage 00/NCBI MD5.

For a record-limited pilot, the stream ends early by design, so the source hash is marked `partial_stream` and is **not** treated as verified.

In [17]:
class HashingReader:
    """Transparent compressed-byte hashing wrapper for an HTTP raw stream."""

    def __init__(self, raw):
        self.raw = raw
        self.md5 = hashlib.md5()
        self.sha256 = hashlib.sha256()
        self.bytes_read = 0

    def read(self, size=-1):
        chunk = self.raw.read(size)
        if chunk:
            self.md5.update(chunk)
            self.sha256.update(chunk)
            self.bytes_read += len(chunk)
        return chunk

    def readinto(self, b):
        chunk = self.read(len(b))
        n = len(chunk)
        b[:n] = chunk
        return n

    def readable(self):
        return True

    def seekable(self):
        return False

    def __getattr__(self, name):
        return getattr(self.raw, name)

def source_expected_md5(row):
    val = row.get("ncbi_md5")
    if val is None:
        return None
    if not isinstance(val, str) and pd.isna(val):
        return None
    val = clean_text(val)
    return val.lower() if val else None

print("Production streaming hash wrapper loaded.")

Production streaming hash wrapper loaded.


## 16. Release parser and memory control

Record boundaries:
- VCV XML: `VariationArchive`
- RCV XML: `ClinVarSet`

Each completed record is parsed, normalized, written to bounded buffers, and cleared from the XML tree immediately.

In [18]:
def release_partition_dir(release_month, data_model, source_format):
    return (
        DATA_DIR
        / f"release_month={release_month}"
        / f"data_model={data_model}"
        / f"format={source_format}"
    )

def remove_incomplete_partition(release_dir: Path):
    if not release_dir.exists():
        return

    complete = release_dir / "release_complete.json"
    if complete.exists():
        return

    if DELETE_INCOMPLETE_PARTITION_BEFORE_RETRY:
        shutil.rmtree(release_dir)

def parse_stream_source(row: pd.Series, record_limit=None):
    release_month = str(row["release_month"])
    data_model = str(row["data_model"]).upper()
    source_format = str(row["format_generation"]).lower()
    source_url = str(row["url"])
    expected_md5 = source_expected_md5(row)

    if record_limit is not None:
        raise ValueError("Production parser requires record_limit=None.")

    if not expected_md5:
        raise RuntimeError(
            f"Missing frozen NCBI MD5 for {release_month} {data_model}"
        )

    release_dir = release_partition_dir(
        release_month, data_model, source_format
    )
    complete_path = release_dir / "release_complete.json"

    if RESUME_COMPLETED_RELEASES and complete_path.exists():
        prior = json.loads(complete_path.read_text())
        if prior.get("md5_verification_status") != "verified":
            raise RuntimeError(
                f"Completion marker exists but MD5 was not verified: {complete_path}"
            )
        print("SKIP verified completed source:", release_month, data_model)
        return prior

    remove_incomplete_partition(release_dir)
    release_dir.mkdir(parents=True, exist_ok=True)

    writers = make_writers(release_dir)
    record_tag = "VariationArchive" if data_model == "VCV" else "ClinVarSet"

    counters = Counter()
    started = time.time()
    parse_error = None
    response = None
    hashing_raw = None
    source_session = None

    try:
        # Requests Session objects are not shared across worker threads.
        source_session = requests.Session()
        source_session.headers.update({"User-Agent": USER_AGENT})

        response = source_session.get(
            source_url,
            stream=True,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        response.raw.decode_content = False

        hashing_raw = HashingReader(response.raw)

        with gzip.GzipFile(fileobj=hashing_raw, mode="rb") as gz:
            context = etree.iterparse(
                gz,
                events=("end",),
                recover=True,
                huge_tree=True,
            )

            for _, elem in context:
                if localname(elem.tag) != record_tag:
                    continue

                if data_model == "VCV":
                    row_vcv, links, scvs = parse_vcv_record(
                        elem=elem,
                        release_month=release_month,
                        source_url=source_url,
                        source_format=source_format,
                    )

                    if row_vcv is not None:
                        writers["vcv_state"].add(row_vcv)
                        writers["vcv_rcv_link"].add_many(links)
                        writers["scv_state"].add_many(scvs)

                        counters["aggregate_records"] += 1
                        counters["scv_records"] += len(scvs)
                        counters["vcv_rcv_links"] += len(links)

                else:
                    row_rcv, scvs = parse_rcv_record(
                        elem=elem,
                        release_month=release_month,
                        source_url=source_url,
                        source_format=source_format,
                    )

                    if row_rcv is not None:
                        writers["rcv_state"].add(row_rcv)
                        writers["scv_state"].add_many(scvs)

                        counters["aggregate_records"] += 1
                        counters["scv_records"] += len(scvs)

                clear_element(elem)

                if counters["aggregate_records"] % 100_000 == 0:
                    elapsed = max(time.time() - started, 1)
                    compressed_gib = hashing_raw.bytes_read / (1024**3)
                    print(
                        f"{release_month} {data_model}: "
                        f"{counters['aggregate_records']:,} aggregates | "
                        f"{counters['scv_records']:,} SCVs | "
                        f"{compressed_gib:,.2f} GiB compressed read | "
                        f"{counters['aggregate_records']/elapsed:,.0f} records/s"
                    )

        # Reaching here means the gzip/XML stream reached EOF.
        for w in writers.values():
            w.close()

        computed_md5 = hashing_raw.md5.hexdigest()
        computed_sha256 = hashing_raw.sha256.hexdigest()
        compressed_bytes_read = hashing_raw.bytes_read

        md5_status = (
            "verified"
            if computed_md5 == expected_md5
            else "mismatch"
        )

        elapsed = time.time() - started

        qc = {
            "release_month": release_month,
            "data_model": data_model,
            "source_format": source_format,
            "source_url": source_url,
            "expected_ncbi_md5": expected_md5,
            "computed_stream_md5": computed_md5,
            "computed_stream_sha256": computed_sha256,
            "compressed_bytes_read": int(compressed_bytes_read),
            "compressed_gib_read": float(compressed_bytes_read / (1024**3)),
            "stream_complete": True,
            "record_limit": None,
            "aggregate_records": int(counters["aggregate_records"]),
            "scv_records": int(counters["scv_records"]),
            "vcv_rcv_links": int(counters["vcv_rcv_links"]),
            "elapsed_seconds": float(elapsed),
            "parse_error": None,
            "md5_verification_status": md5_status,
            "parquet_rows": {
                name: int(w.total_rows)
                for name, w in writers.items()
            },
            "completed_utc": datetime.now(timezone.utc).isoformat(),
        }

        (release_dir / "release_qc.json").write_text(
            json.dumps(qc, indent=2),
            encoding="utf-8",
        )

        if md5_status != "verified":
            raise AssertionError(
                f"MD5 mismatch for {release_month} {data_model}: "
                f"expected {expected_md5}, got {computed_md5}"
            )

        # Completion marker is written only after full-stream MD5 verification.
        complete_path.write_text(
            json.dumps(qc, indent=2),
            encoding="utf-8",
        )

        return qc

    except Exception as e:
        parse_error = repr(e)
        for w in writers.values():
            try:
                w.close()
            except Exception:
                pass

        failure = {
            "release_month": release_month,
            "data_model": data_model,
            "source_format": source_format,
            "source_url": source_url,
            "error": parse_error,
            "failed_utc": datetime.now(timezone.utc).isoformat(),
        }

        (release_dir / "release_failed.json").write_text(
            json.dumps(failure, indent=2),
            encoding="utf-8",
        )
        raise

    finally:
        if response is not None:
            response.close()
        if source_session is not None:
            source_session.close()

print("Production full-stream release parser loaded.")

Production full-stream release parser loaded.


## 17. Source-level retry wrapper

A source is retried from the beginning after transient network/parser failure. Incomplete Parquet shards are removed before retry so a failed source cannot silently duplicate records.

In [19]:
def process_source_with_retry(row, record_limit=None):
    last_error = None

    for attempt in range(1, MAX_SOURCE_ATTEMPTS + 1):
        try:
            print(
                f"\n=== {row['release_month']} | "
                f"{row['data_model']} | {row['format_generation']} | "
                f"attempt {attempt}/{MAX_SOURCE_ATTEMPTS} ==="
            )
            return parse_stream_source(row, record_limit=record_limit)

        except Exception as e:
            last_error = e
            print("Source attempt failed:", repr(e))

            if attempt < MAX_SOURCE_ATTEMPTS:
                wait = RETRY_BACKOFF_SECONDS * attempt
                print(f"Retrying in {wait}s ...")
                time.sleep(wait)

    raise RuntimeError(
        f"Source failed after {MAX_SOURCE_ATTEMPTS} attempts: "
        f"{row['release_month']} {row['data_model']} — {last_error!r}"
    )

In [20]:
import time  # defensive import for standalone reruns

run_results = []
run_failures = []

batch_started = time.time()

months = list(
    pd.period_range(
        BATCH_START_MONTH,
        BATCH_END_MONTH,
        freq="M",
    ).astype(str)
)

print("Execution strategy: RCV ∥ VCV within month")
print("Maximum concurrent source streams:", MAX_PARALLEL_SOURCES)
print("Months:", months)

for month_index, month in enumerate(months, start=1):
    month_sources = (
        selected[selected["release_month"].astype(str) == month]
        .sort_values("data_model")
        .reset_index(drop=True)
    )

    if len(month_sources) != len(MODELS_TO_PROCESS):
        raise RuntimeError(
            f"{month}: expected {len(MODELS_TO_PROCESS)} sources, "
            f"found {len(month_sources)}"
        )

    print(
        f"\n{'#' * 78}\n"
        f"MONTH {month_index}/{len(months)} — {month} — "
        f"starting {len(month_sources)} concurrent sources\n"
        f"{'#' * 78}"
    )

    month_failures = []

    with cf.ThreadPoolExecutor(
        max_workers=MAX_PARALLEL_SOURCES,
        thread_name_prefix=f"ges3_{month}",
    ) as executor:

        future_to_row = {}

        for _, source_row in month_sources.iterrows():
            future = executor.submit(
                process_source_with_retry,
                source_row.copy(),
                None,
            )
            future_to_row[future] = source_row.copy()

        for future in cf.as_completed(future_to_row):
            source_row = future_to_row[future]

            try:
                result = future.result()
                run_results.append(result)

                print(
                    "\nCOMPLETED:",
                    result["release_month"],
                    result["data_model"],
                    "| aggregates =", f"{result['aggregate_records']:,}",
                    "| SCVs =", f"{result['scv_records']:,}",
                    "| MD5 =", result["md5_verification_status"],
                    "| minutes =", f"{result['elapsed_seconds']/60:.1f}",
                )

            except Exception as e:
                failure = {
                    "release_month": str(source_row["release_month"]),
                    "data_model": str(source_row["data_model"]),
                    "format_generation": str(source_row["format_generation"]),
                    "url": str(source_row["url"]),
                    "error": repr(e),
                }
                run_failures.append(failure)
                month_failures.append(failure)
                print("\nFINAL SOURCE FAILURE:", failure)

    # Fail-fast between months, but let the paired source finish cleanly first.
    if month_failures:
        print(
            f"\nStopping before the next month because {month} "
            f"had {len(month_failures)} failed source(s)."
        )
        break

    print(f"\nMONTH COMPLETE: {month}")

results_df = pd.DataFrame(run_results)
failures_df = pd.DataFrame(run_failures)

batch_elapsed = time.time() - batch_started

print("\nBatch elapsed hours:", round(batch_elapsed / 3600, 3))
print(
    "Sum of individual source hours:",
    round(
        sum(float(r.get("elapsed_seconds", 0)) for r in run_results) / 3600,
        3,
    ),
)
if run_results:
    serial_equivalent = sum(
        float(r.get("elapsed_seconds", 0)) for r in run_results
    )
    print(
        "Observed concurrency wall-clock ratio:",
        round(batch_elapsed / serial_equivalent, 3)
        if serial_equivalent > 0 else None,
    )

display(results_df)

if len(failures_df):
    print("\nFailures:")
    display(failures_df)

Execution strategy: RCV ∥ VCV within month
Maximum concurrent source streams: 2
Months: ['2021-10', '2021-11', '2021-12']

##############################################################################
MONTH 1/3 — 2021-10 — starting 2 concurrent sources
##############################################################################

=== 2021-10 | RCV | legacy | attempt 1/3 ===

=== 2021-10 | VCV | legacy | attempt 1/3 ===
SKIP verified completed source: 2021-10 VCV

COMPLETED: 2021-10 VCV | aggregates = 1,158,803 | SCVs = 1,823,184 | MD5 = verified | minutes = 58.2
SKIP verified completed source: 2021-10 RCV

COMPLETED: 2021-10 RCV | aggregates = 1,594,206 | SCVs = 1,823,196 | MD5 = verified | minutes = 64.2

MONTH COMPLETE: 2021-10

##############################################################################
MONTH 2/3 — 2021-11 — starting 2 concurrent sources
##############################################################################

=== 2021-11 | RCV | legacy | attempt 1/3 ===



,release_month,data_model,source_format,source_url,expected_ncbi_md5,computed_stream_md5,computed_stream_sha256,compressed_bytes_read,compressed_gib_read,stream_complete,record_limit,aggregate_records,scv_records,vcv_rcv_links,elapsed_seconds,parse_error,md5_verification_status,parquet_rows,completed_utc
0,2021-10,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,ef02a76001cea1e6fb54ecaadc075ce4,ef02a76001cea1e6fb54ecaadc075ce4,36bba70b63462c01b850ea9c53fd12d2ead38cf7025e11...,1539551851,1.433819,True,None,1158803,1823184,1594191,3492.703681,None,verified,"{'vcv_state': 1158803, 'rcv_state': 0, 'scv_st...",2026-08-17T04:57:05.382019+00:00
1,2021-10,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,c67eed9a99a1c33ce335c00651a0e6af,c67eed9a99a1c33ce335c00651a0e6af,1ebcae71a2a56b8f564fd0dcd2a392a1f1c9e547a38ea8...,1698775795,1.582108,True,None,1594206,1823196,0,3854.133998,None,verified,"{'vcv_state': 0, 'rcv_state': 1594206, 'scv_st...",2026-08-17T05:03:06.815846+00:00
2,2021-11,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,f9f0e6b48cea62c05cbc2923629c6163,f9f0e6b48cea62c05cbc2923629c6163,275cbcd620e361a571e1cf3ab4a72d006b7e2cc5cfd215...,1717730058,1.599761,True,None,1162402,1847008,1600911,3583.619689,None,verified,"{'vcv_state': 1162402, 'rcv_state': 0, 'scv_st...",2026-08-17T06:02:50.485415+00:00
3,2021-11,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,f3a84e95f61747a702eb87fc3912a3ff,f3a84e95f61747a702eb87fc3912a3ff,0d1ed2917360f980fa26d50c45d34f659fca7d3b14d01a...,1777365024,1.655300,True,None,1600916,1847012,0,3939.242896,None,verified,"{'vcv_state': 0, 'rcv_state': 1600916, 'scv_st...",2026-08-17T06:08:46.105891+00:00
4,2021-12,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,ea0a3f142e0f7967e43d7e1a0cccb689,ea0a3f142e0f7967e43d7e1a0cccb689,f5fa7c5c67d6b713b79260a72285f7d5a77ada01832ae7...,1751734181,1.631430,True,None,1184702,1883179,1628401,3719.819558,None,verified,"{'vcv_state': 1184702, 'rcv_state': 0, 'scv_st...",2026-08-17T19:23:42.154553+00:00
5,2021-12,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,b14249d9184a22989a2b3c4e17bd04e3,b14249d9184a22989a2b3c4e17bd04e3,ec5578cdfda6b451524b9bfcda4838767f9a9ff621087c...,1787489826,1.664730,True,None,1628400,1883185,0,4109.080335,None,verified,"{'vcv_state': 0, 'rcv_state': 1628400, 'scv_st...",2026-08-17T19:30:10.709815+00:00


## 19. Memory-bounded production QC

The pilot notebook could load 50,000-row partitions into pandas. A full ClinVar release may contain far more rows, so that approach is unsafe for production.

This QC scans Parquet files in **bounded Arrow batches**. It validates the complete partition without concatenating the entire release into RAM.

In [21]:
import pyarrow.dataset as ds

def parquet_files(release_dir: Path, table_name: str):
    return sorted((release_dir / table_name).glob("*.parquet"))

def parquet_row_count(release_dir: Path, table_name: str) -> int:
    total = 0
    for p in parquet_files(release_dir, table_name):
        total += pq.ParquetFile(p).metadata.num_rows
    return int(total)

def scan_column_qc(
    release_dir: Path,
    table_name: str,
    accession_col: str,
    accession_prefix: str,
):
    files = parquet_files(release_dir, table_name)

    if not files:
        return {
            "rows_scanned": 0,
            "bad_accessions": 0,
            "missing_release_month": 0,
            "missing_source_url": 0,
        }

    dataset = ds.dataset([str(p) for p in files], format="parquet")

    counts = Counter()

    scanner = dataset.scanner(
        columns=[accession_col, "release_month", "source_url"],
        batch_size=100_000,
    )

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        counts["rows_scanned"] += len(df)
        counts["bad_accessions"] += int(
            (~df[accession_col].fillna("").str.startswith(accession_prefix)).sum()
        )
        counts["missing_release_month"] += int(
            df["release_month"].isna().sum()
        )
        counts["missing_source_url"] += int(
            df["source_url"].isna().sum()
        )

    return dict(counts)

def scan_scv_qc(release_dir: Path):
    files = parquet_files(release_dir, "scv_state")

    if not files:
        return {
            "rows_scanned": 0,
            "bad_scv_accessions": 0,
        }

    dataset = ds.dataset([str(p) for p in files], format="parquet")
    counts = Counter()

    scanner = dataset.scanner(
        columns=["scv_accession"],
        batch_size=100_000,
    )

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        counts["rows_scanned"] += len(df)
        counts["bad_scv_accessions"] += int(
            (~df["scv_accession"].fillna("").str.startswith("SCV")).sum()
        )

    return dict(counts)

qc_rows = []

for result in run_results:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    if result["data_model"] == "VCV":
        table_name = "vcv_state"
        accession_col = "vcv_accession"
        prefix = "VCV"
    else:
        table_name = "rcv_state"
        accession_col = "rcv_accession"
        prefix = "RCV"

    agg_qc = scan_column_qc(
        release_dir, table_name, accession_col, prefix
    )
    scv_qc = scan_scv_qc(release_dir)

    aggregate_rows = parquet_row_count(release_dir, table_name)
    scv_rows = parquet_row_count(release_dir, "scv_state")

    parser_count_match = (
        aggregate_rows == int(result["aggregate_records"])
        and scv_rows == int(result["scv_records"])
    )

    completion_marker = release_dir / "release_complete.json"

    qc_rows.append({
        "release_month": result["release_month"],
        "data_model": result["data_model"],
        "source_format": result["source_format"],
        "aggregate_rows": aggregate_rows,
        "scv_rows": scv_rows,
        "bad_aggregate_accessions": int(agg_qc.get("bad_accessions", 0)),
        "bad_scv_accessions": int(scv_qc.get("bad_scv_accessions", 0)),
        "missing_release_month": int(agg_qc.get("missing_release_month", 0)),
        "missing_source_url": int(agg_qc.get("missing_source_url", 0)),
        "parser_count_match": bool(parser_count_match),
        "md5_verified": result["md5_verification_status"] == "verified",
        "completion_marker_exists": completion_marker.exists(),
    })

artifact_qc = pd.DataFrame(qc_rows)
display(artifact_qc)

if len(artifact_qc):
    all_structural_ok = bool(
        (artifact_qc["bad_aggregate_accessions"] == 0).all()
        and (artifact_qc["bad_scv_accessions"] == 0).all()
        and (artifact_qc["missing_release_month"] == 0).all()
        and (artifact_qc["missing_source_url"] == 0).all()
        and artifact_qc["parser_count_match"].all()
        and artifact_qc["md5_verified"].all()
        and artifact_qc["completion_marker_exists"].all()
    )
else:
    all_structural_ok = False

print("ALL PRODUCTION STRUCTURAL QC PASSED:", all_structural_ok)

,release_month,data_model,source_format,aggregate_rows,scv_rows,bad_aggregate_accessions,bad_scv_accessions,missing_release_month,missing_source_url,parser_count_match,md5_verified,completion_marker_exists
0,2021-10,VCV,legacy,1158803,1823184,0,0,0,0,True,True,True
1,2021-10,RCV,legacy,1594206,1823196,0,0,0,0,True,True,True
2,2021-11,VCV,legacy,1162402,1847008,0,0,0,0,True,True,True
3,2021-11,RCV,legacy,1600916,1847012,0,0,0,0,True,True,True
4,2021-12,VCV,legacy,1184702,1883179,0,0,0,0,True,True,True
5,2021-12,RCV,legacy,1628400,1883185,0,0,0,0,True,True,True


ALL PRODUCTION STRUCTURAL QC PASSED: True


## 20. Memory-bounded classification semantics audit

The entire classification columns are scanned in batches. Legacy releases must retain aggregate interpretation in `legacy_*` fields rather than being silently rewritten as modern classification axes.

In [22]:
def scan_classification_counts(release_dir: Path, table_name: str):
    files = parquet_files(release_dir, table_name)
    cols = [
        "germline_description",
        "somatic_clinical_impact_description",
        "oncogenicity_description",
        "legacy_classification_description",
        "legacy_germline_candidate",
    ]

    counts = Counter()

    if not files:
        return counts

    dataset = ds.dataset([str(p) for p in files], format="parquet")
    scanner = dataset.scanner(columns=cols, batch_size=100_000)

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        counts["rows"] += len(df)
        counts["germline_nonnull"] += int(df["germline_description"].notna().sum())
        counts["somatic_impact_nonnull"] += int(
            df["somatic_clinical_impact_description"].notna().sum()
        )
        counts["oncogenicity_nonnull"] += int(
            df["oncogenicity_description"].notna().sum()
        )
        counts["legacy_nonnull"] += int(
            df["legacy_classification_description"].notna().sum()
        )
        counts["legacy_germline_candidate"] += int(
            df["legacy_germline_candidate"].fillna(False).sum()
        )

    return counts

classification_audit_rows = []

for result in run_results:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    table_name = (
        "vcv_state" if result["data_model"] == "VCV"
        else "rcv_state"
    )

    c = scan_classification_counts(release_dir, table_name)

    classification_audit_rows.append({
        "release_month": result["release_month"],
        "data_model": result["data_model"],
        "source_format": result["source_format"],
        **{k: int(v) for k, v in c.items()},
    })

classification_audit = pd.DataFrame(classification_audit_rows)
display(classification_audit)

# Batch 4 is historical/legacy. It should not silently populate modern aggregate axes.
legacy_rows = classification_audit[
    classification_audit["source_format"] == "legacy"
]

legacy_semantics_ok = True
if len(legacy_rows):
    legacy_semantics_ok = bool(
        (legacy_rows.get("germline_nonnull", 0) == 0).all()
        and (legacy_rows.get("somatic_impact_nonnull", 0) == 0).all()
        and (legacy_rows.get("oncogenicity_nonnull", 0) == 0).all()
    )

print("LEGACY CLASSIFICATION-AXIS SEPARATION PASSED:", legacy_semantics_ok)

,release_month,data_model,source_format,rows,germline_nonnull,somatic_impact_nonnull,oncogenicity_nonnull,legacy_nonnull,legacy_germline_candidate
0,2021-10,VCV,legacy,1158803,0,0,0,1158137,1069167
1,2021-10,RCV,legacy,1594206,0,0,0,1594206,1552261
2,2021-11,VCV,legacy,1162402,0,0,0,1161735,1071988
3,2021-11,RCV,legacy,1600916,0,0,0,1600916,1558456
4,2021-12,VCV,legacy,1184702,0,0,0,1184045,1093002
5,2021-12,RCV,legacy,1628400,0,0,0,1628400,1584875


LEGACY CLASSIFICATION-AXIS SEPARATION PASSED: True


## 21. Memory-bounded RCV → SCV relationship audit

Every RCV-derived SCV should retain its parent RCV accession. Submitter distinct counts are calculated incrementally without loading all SCVs at once.

In [23]:
def scan_rcv_scv_relationship(release_dir: Path):
    files = parquet_files(release_dir, "scv_state")
    if not files:
        return {
            "scv_rows": 0,
            "scv_with_parent_rcv": 0,
            "unique_parent_rcv": 0,
            "unique_submitters": 0,
        }

    dataset = ds.dataset([str(p) for p in files], format="parquet")
    scanner = dataset.scanner(
        columns=["parent_rcv_accession", "submitter_name"],
        batch_size=100_000,
    )

    n_rows = 0
    n_parent = 0
    parent_ids = set()
    submitters = set()

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        n_rows += len(df)
        n_parent += int(df["parent_rcv_accession"].notna().sum())
        parent_ids.update(
            x for x in df["parent_rcv_accession"].dropna().astype(str)
        )
        submitters.update(
            x for x in df["submitter_name"].dropna().astype(str)
        )

    return {
        "scv_rows": n_rows,
        "scv_with_parent_rcv": n_parent,
        "unique_parent_rcv": len(parent_ids),
        "unique_submitters": len(submitters),
    }

relationship_rows = []

for result in run_results:
    if result["data_model"] != "RCV":
        continue

    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    s = scan_rcv_scv_relationship(release_dir)

    relationship_rows.append({
        "release_month": result["release_month"],
        "source_format": result["source_format"],
        **{k: int(v) for k, v in s.items()},
        "all_scvs_have_parent_rcv": (
            s["scv_rows"] == s["scv_with_parent_rcv"]
        ),
    })

relationship_audit = pd.DataFrame(relationship_rows)
display(relationship_audit)

rcv_scv_relationship_ok = bool(
    relationship_audit["all_scvs_have_parent_rcv"].all()
) if len(relationship_audit) else False

print("RCV→SCV RELATIONSHIP QC PASSED:", rcv_scv_relationship_ok)

,release_month,source_format,scv_rows,scv_with_parent_rcv,unique_parent_rcv,unique_submitters,all_scvs_have_parent_rcv
0,2021-10,legacy,1823196,1823196,1594206,2042,True
1,2021-11,legacy,1847012,1847012,1600916,2068,True
2,2021-12,legacy,1883185,1883185,1628400,2086,True


RCV→SCV RELATIONSHIP QC PASSED: True


## 22. Data dictionary

Freeze a compact data dictionary now so later feature notebooks do not reinterpret columns ad hoc.

In [24]:
data_dictionary = {
    "vcv_state": {
        "unit": "VCV aggregate state in one ClinVar monthly release",
        "identity": ["release_month", "vcv_accession"],
        "notes": (
            "Current classification axes are separate. Legacy single "
            "classification remains explicitly legacy."
        ),
    },
    "rcv_state": {
        "unit": "RCV variant-condition aggregate state in one monthly release",
        "identity": ["release_month", "rcv_accession"],
        "notes": (
            "Primary GES 3.0 longitudinal prediction-unit foundation."
        ),
    },
    "scv_state": {
        "unit": "Submitted ClinVar assertion visible in one monthly release",
        "identity": [
            "release_month",
            "source_data_model",
            "scv_accession",
            "parent_rcv_accession",
            "parent_vcv_accession",
        ],
        "notes": (
            "RCV-derived SCV rows preserve condition-context relationship. "
            "VCV-derived SCVs may overlap the same SCV accession and must not "
            "be naively double-counted downstream."
        ),
    },
    "vcv_rcv_link": {
        "unit": "VCV-to-RCV link visible in one monthly VCV release",
        "identity": [
            "release_month",
            "vcv_accession",
            "rcv_accession",
        ],
    },
    "classification_policy": {
        "current": (
            "Preserve germline, somatic clinical impact, and oncogenicity "
            "as separate fields."
        ),
        "legacy": (
            "Preserve single classification as legacy_*; "
            "legacy_germline_candidate is only a conservative term flag, "
            "not final cohort eligibility."
        ),
    },
}

data_dictionary_path = META_DIR / "stage01_data_dictionary.json"
data_dictionary_path.write_text(
    json.dumps(data_dictionary, indent=2),
    encoding="utf-8",
)

print(json.dumps(data_dictionary, indent=2))

{
  "vcv_state": {
    "unit": "VCV aggregate state in one ClinVar monthly release",
    "identity": [
      "release_month",
      "vcv_accession"
    ],
    "notes": "Current classification axes are separate. Legacy single classification remains explicitly legacy."
  },
  "rcv_state": {
    "unit": "RCV variant-condition aggregate state in one monthly release",
    "identity": [
      "release_month",
      "rcv_accession"
    ],
    "notes": "Primary GES 3.0 longitudinal prediction-unit foundation."
  },
  "scv_state": {
    "unit": "Submitted ClinVar assertion visible in one monthly release",
    "identity": [
      "release_month",
      "source_data_model",
      "scv_accession",
      "parent_rcv_accession",
      "parent_vcv_accession"
    ],
    "notes": "RCV-derived SCV rows preserve condition-context relationship. VCV-derived SCVs may overlap the same SCV accession and must not be naively double-counted downstream."
  },
  "vcv_rcv_link": {
    "unit": "VCV-to-RCV link visib

## 23. Freeze Production Batch 4 metadata and artifact hashes

All metadata/QC files are SHA-256 hashed. Parquet shards are also hashed by default.

This can take additional time because hashing rereads the normalized files, but it preserves the reproducibility discipline required by the GES 3.0 blueprint.

In [25]:
def sha256_file(path: Path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

results_path = QC_DIR / "stage01_batch4_2021Q4_results.csv"
failures_path = QC_DIR / "stage01_batch4_2021Q4_failures.csv"
artifact_qc_path = QC_DIR / "stage01_batch4_2021Q4_artifact_qc.csv"
class_qc_path = QC_DIR / "stage01_batch4_2021Q4_classification_audit.csv"
relation_qc_path = QC_DIR / "stage01_batch4_2021Q4_rcv_scv_audit.csv"

results_df.to_csv(results_path, index=False)
failures_df.to_csv(failures_path, index=False)
artifact_qc.to_csv(artifact_qc_path, index=False)
classification_audit.to_csv(class_qc_path, index=False)
relationship_audit.to_csv(relation_qc_path, index=False)

runtime_metadata = {
    "run_utc": RUN_UTC,
    "run_profile": RUN_PROFILE,
    "batch_start_month": BATCH_START_MONTH,
    "batch_end_month": BATCH_END_MONTH,
    "manifest_origin": manifest_origin,
    "stage00_manifest_sha256": manifest_sha256,
    "batch_run_manifest_sha256": run_manifest_sha256,
    "record_limit_per_source": None,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pa.__version__,
    "lxml": lxml.__version__,
    "requests": requests.__version__,
    "selected_sources": int(len(selected)),
    "successful_sources": int(len(run_results)),
    "failed_sources": int(len(run_failures)),
    "all_structural_qc_passed": bool(all_structural_ok),
    "legacy_semantics_ok": bool(legacy_semantics_ok),
    "rcv_scv_relationship_ok": bool(rcv_scv_relationship_ok),
    "batch_elapsed_seconds": float(batch_elapsed),
    "execution_strategy": "two_source_parallel_within_month",
    "max_parallel_sources": int(MAX_PARALLEL_SOURCES),
    "detected_gpu_names": DETECTED_GPU_NAMES,
    "host_logical_cpu_count": HOST_CPU_COUNT,
    "sum_individual_source_seconds": float(
        sum(float(r.get("elapsed_seconds", 0)) for r in run_results)
    ),
}

runtime_path = META_DIR / "stage01_batch4_2021Q4_runtime_metadata.json"
runtime_path.write_text(
    json.dumps(runtime_metadata, indent=2),
    encoding="utf-8",
)

hash_rows = []

# Hash ONLY Batch-3 small artifacts, preventing cross-batch metadata
# from leaking into this batch's hash manifest.
current_small_artifacts = [
    run_manifest_path,
    META_DIR / "stage01_batch4_2021Q4_run_manifest.sha256",
    data_dictionary_path,
    results_path,
    failures_path,
    artifact_qc_path,
    class_qc_path,
    relation_qc_path,
    runtime_path,
]

for p in current_small_artifacts:
    p = Path(p)
    if not p.exists():
        continue
    hash_rows.append({
        "relative_path": str(p.relative_to(STAGE01_DIR)),
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

# Hash normalized Parquet shards only for this batch.
if HASH_PARQUET_SHARDS:
    batch_months = set(
        pd.period_range(
            BATCH_START_MONTH, BATCH_END_MONTH, freq="M"
        ).astype(str)
    )

    for p in DATA_DIR.rglob("*.parquet"):
        if not any(
            f"release_month={m}" in str(p)
            for m in batch_months
        ):
            continue

        hash_rows.append({
            "relative_path": str(p.relative_to(STAGE01_DIR)),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows).sort_values("relative_path")
hash_path = META_DIR / "stage01_batch4_2021Q4_artifact_sha256.csv"
hash_df.to_csv(hash_path, index=False)

print(json.dumps(runtime_metadata, indent=2))
print("\nHashed artifacts:", len(hash_df))
display(hash_df.head(20))

{
  "run_utc": "2026-08-17T18:21:07.157312+00:00",
  "run_profile": "PRODUCTION_BATCH",
  "batch_start_month": "2021-10",
  "batch_end_month": "2021-12",
  "manifest_origin": "frozen_csv:/content/drive/MyDrive/GES3/stage00/clinvar_release_manifest_canonical.csv",
  "stage00_manifest_sha256": "8b483f722e7271d51fe9b343df3a385804f2ff48c54701a13311f6c87b246c5f",
  "batch_run_manifest_sha256": "1d9aa38ce60c5e98fcd8e0051ff7b5e8905fad9994d819170779ec58b095b9e6",
  "record_limit_per_source": null,
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "pandas": "2.2.2",
  "numpy": "2.0.2",
  "pyarrow": "18.1.0",
  "lxml": "6.1.1",
  "requests": "2.32.4",
  "selected_sources": 6,
  "successful_sources": 6,
  "failed_sources": 0,
  "all_structural_qc_passed": true,
  "legacy_semantics_ok": true,
  "rcv_scv_relationship_ok": true,
  "batch_elapsed_seconds": 4115.427150249481,
  "execution_strategy": "two_source_parallel_within_mon

,relative_path,bytes,sha256
0,metadata/stage01_batch4_2021Q4_run_manifest.csv,2315,1d9aa38ce60c5e98fcd8e0051ff7b5e8905fad9994d819...
1,metadata/stage01_batch4_2021Q4_run_manifest.sh...,105,a04de5c93be46fa05206a9f068ce8bba654643d705254a...
8,metadata/stage01_batch4_2021Q4_runtime_metadat...,1180,3c1775ca3cc356e6499e87344a9c0948fa2ae4616bcc3d...
2,metadata/stage01_data_dictionary.json,1412,01835d293e0c7d04eba25f0c056df65c44cfe06bb86378...
102,normalized/release_month=2021-10/data_model=RC...,1094023,e83c11f6112feb87baf64ea540f617369e0c90b4ffada4...
103,normalized/release_month=2021-10/data_model=RC...,930968,e5f2296ecb5e3fa3406274db288d602fa18002f8125eef...
104,normalized/release_month=2021-10/data_model=RC...,1082045,71a6fb62d8bacbfbc8ff11cf246c6716456f0a0aa7c256...
105,normalized/release_month=2021-10/data_model=RC...,1105539,da68315687b12b9d5d3287ba5d0db446f776ddef8c3514...
106,normalized/release_month=2021-10/data_model=RC...,923047,01adeb4d5c71a9dddc3c90358e37b5108126f257285461...
107,normalized/release_month=2021-10/data_model=RC...,1075425,5498cffc1987fe42a2811ca41ce95c431f464f0ce9cc77...


## 24. Batch 4 go / no-go decision

Batch 4 is accepted only if **all six sources** are complete and all production invariants pass:

- 6/6 selected sources successful
- 0 source failures
- full-stream NCBI MD5 verified for every source
- completion marker present for every source
- Parquet row counts match parser counts
- accession/lineage structural QC passes
- legacy/current classification semantics remain separated
- all RCV-derived SCVs retain a parent RCV

A partial batch is useful for resumption, but it is **not yet accepted as a completed production batch**.

In [26]:
expected_source_keys = {
    (str(r["release_month"]), str(r["data_model"]))
    for _, r in selected.iterrows()
}

completed_source_keys = {
    (str(r["release_month"]), str(r["data_model"]))
    for r in run_results
    if r.get("md5_verification_status") == "verified"
    and r.get("stream_complete") is True
}

all_six_completed = completed_source_keys == expected_source_keys

decision = {
    "stage": "GES3_01_production_batch4",
    "batch": f"{BATCH_START_MONTH}_to_{BATCH_END_MONTH}",
    "expected_sources": len(expected_source_keys),
    "verified_completed_sources": len(completed_source_keys),
    "failed_sources": len(run_failures),
    "all_six_completed": bool(all_six_completed),
    "structural_qc_passed": bool(all_structural_ok),
    "legacy_semantics_ok": bool(legacy_semantics_ok),
    "rcv_scv_relationship_ok": bool(rcv_scv_relationship_ok),
    "execution_strategy": "two_source_parallel_within_month",
    "max_parallel_sources": int(MAX_PARALLEL_SOURCES),
    "detected_gpu_names": DETECTED_GPU_NAMES,
    "batch_elapsed_hours": float(batch_elapsed / 3600),
}

decision["batch4_go"] = bool(
    all_six_completed
    and len(run_failures) == 0
    and all_structural_ok
    and legacy_semantics_ok
    and rcv_scv_relationship_ok
)

decision_path = QC_DIR / "stage01_batch4_2021Q4_decision.json"
decision_path.write_text(
    json.dumps(decision, indent=2),
    encoding="utf-8",
)

print(json.dumps(decision, indent=2))

{
  "stage": "GES3_01_production_batch4",
  "batch": "2021-10_to_2021-12",
  "expected_sources": 6,
  "verified_completed_sources": 6,
  "failed_sources": 0,
  "all_six_completed": true,
  "structural_qc_passed": true,
  "legacy_semantics_ok": true,
  "rcv_scv_relationship_ok": true,
  "execution_strategy": "two_source_parallel_within_month",
  "max_parallel_sources": 2,
  "detected_gpu_names": [
    "NVIDIA A100-SXM4-80GB"
  ],
  "batch_elapsed_hours": 1.1431742084026337,
  "batch4_go": true
}


## 25. Create compact Batch 4 metadata/QC bundle

The normalized Parquet files remain in persistent Drive storage. The ZIP contains only the reproducibility artifacts needed for GitHub review:
- frozen batch manifest
- runtime metadata
- QC tables
- decision JSON
- data dictionary
- hash manifest

In [27]:
bundle_dir = Path("/content/GES3_STAGE01_BATCH4_2021Q4_METADATA")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

copy_candidates = [
    run_manifest_path,
    data_dictionary_path,
    runtime_path,
    hash_path,
    decision_path,
    STAGE00_DIR / "stage00_production_lock.json",
    STAGE00_DIR / "clinvar_release_manifest_canonical.sha256",
]

for p in copy_candidates:
    p = Path(p)
    if p.exists():
        shutil.copy2(p, bundle_dir / p.name)

for p in QC_DIR.glob("stage01_batch4_2021Q4*"):
    if p.is_file():
        shutil.copy2(p, bundle_dir / p.name)

bundle_zip = shutil.make_archive(
    "/content/GES3_STAGE01_BATCH4_2021Q4_METADATA_QC",
    "zip",
    root_dir=bundle_dir,
)

print("Metadata/QC bundle:", bundle_zip)
print("Bundle bytes:", Path(bundle_zip).stat().st_size)

Metadata/QC bundle: /content/GES3_STAGE01_BATCH4_2021Q4_METADATA_QC.zip
Bundle bytes: 30470


## 26. Small production preview

Only the first Parquet shard of each completed aggregate table is previewed. The full release is **not** loaded into pandas.

In [28]:
for result in run_results:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    table_name = (
        "vcv_state" if result["data_model"] == "VCV"
        else "rcv_state"
    )

    files = parquet_files(release_dir, table_name)

    print(
        "\n",
        result["release_month"],
        result["data_model"],
        result["source_format"],
        table_name,
    )

    if files:
        preview = pd.read_parquet(files[0]).head(5)
        display(preview)
    else:
        print("No Parquet files found.")


 2021-10 VCV legacy vcv_state


,release_month,source_url,source_format,vcv_accession,vcv_version,variation_id,variation_type,variation_name,date_created,date_last_updated,...,oncogenicity_description,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,gene_symbols_json,rcv_accessions_json,citation_ids_json,scv_count
0,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000000441,1,441,Deletion,"BCAM, EX3-4DEL",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2007-03-01,True,"[""BCAM""]","[""RCV000000470""]","[""PubMed:17319831""]",1
1,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003202,1,3202,Insertion,"FKTN, L1 INS",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,1999-11-01,True,"[""FKTN""]","[""RCV000003355""]","[""PubMed:10545611""]",1
2,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003212,1,3212,Deletion,"FKTN, 473-BP DEL, NT5370",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2008-02-01,True,"[""FKTN""]","[""RCV000003367""]","[""PubMed:18177472""]",1
3,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003876,1,3876,Insertion,"HEXB, 18-BP INS",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,1990-10-15,True,"[""HEXB""]","[""RCV000004080""]","[""PubMed:2170400"", ""PubMed:868875""]",1
4,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000004793,1,4793,Deletion,"PRX, 1-BP DEL, 247C",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2002-06-01,True,"[""PRX""]","[""RCV000005060""]","[""PubMed:12112076""]",1



 2021-10 RCV legacy rcv_state


,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000470,4,VCV000000441,1,441,None,None,...,None,Pathogenic,no assertion criteria provided,2007-03-01,True,"[""BLOOD GROUP--LUTHERAN NULL""]","[""OMIM:612773.0003"", ""OMIM:612773.0004"", ""OMIM...",[],"[""PubMed:17319831""]",1
1,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000003355,3,VCV000003202,1,3202,None,None,...,None,Pathogenic,no assertion criteria provided,1999-11-01,True,"[""Congenital muscular dystrophy-dystroglycanop...","[""OMIM:607440.0001"", ""OMIM:607440.0002"", ""OMIM...",[],"[""PubMed:10545611""]",1
2,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000003367,3,VCV000003212,1,3212,None,None,...,None,Pathogenic,no assertion criteria provided,2008-02-01,True,"[""Congenital muscular dystrophy-dystroglycanop...","[""OMIM:607440.0001"", ""OMIM:607440.0002"", ""OMIM...",[],"[""PubMed:18177472""]",1
3,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000004080,2,VCV000003876,1,3876,None,None,...,None,Pathogenic,no assertion criteria provided,1990-10-15,True,"[""Hexosaminidase B (paris)""]","[""MedGen:C4016989""]",[],"[""PubMed:2170400"", ""PubMed:868875""]",1
4,2021-10,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005060,3,VCV000004793,1,4793,None,None,...,None,Pathogenic,no assertion criteria provided,2002-06-01,True,"[""Autosomal recessive Dejerine-Sottas syndrome""]","[""MedGen:CN069172""]",[],"[""PubMed:12112076""]",1



 2021-11 VCV legacy vcv_state


...

,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2021-11,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000470,4,VCV000000441,1,441,None,None,...,None,Pathogenic,no assertion criteria provided,2007-03-01,True,"[""BLOOD GROUP--LUTHERAN NULL""]","[""OMIM:612773.0003"", ""OMIM:612773.0004"", ""OMIM...",[],"[""PubMed:17319831""]",1
1,2021-11,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000004080,2,VCV000003876,1,3876,None,None,...,None,Pathogenic,no assertion criteria provided,1990-10-15,True,"[""Hexosaminidase B (paris)""]","[""MedGen:C4016989""]",[],"[""PubMed:2170400"", ""PubMed:868875""]",1
2,2021-11,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005060,3,VCV000004793,1,4793,None,None,...,None,Pathogenic,no assertion criteria provided,2002-06-01,True,"[""Autosomal recessive Dejerine-Sottas syndrome""]","[""MedGen:CN069172""]",[],"[""PubMed:12112076""]",1
3,2021-11,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005486,4,VCV000005178,1,5178,None,None,...,None,risk factor,no assertion criteria provided,2002-04-01,False,"[""Hypercalciuria, absorptive, susceptibility to""]",[],[],"[""PubMed:11932268""]",1
4,2021-11,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005487,4,VCV000005179,1,5179,None,None,...,None,risk factor,no assertion criteria provided,2002-04-01,False,"[""Hypercalciuria, absorptive, susceptibility to""]",[],[],"[""PubMed:11932268""]",1



 2021-12 VCV legacy vcv_state


,release_month,source_url,source_format,vcv_accession,vcv_version,variation_id,variation_type,variation_name,date_created,date_last_updated,...,oncogenicity_description,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,gene_symbols_json,rcv_accessions_json,citation_ids_json,scv_count
0,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000000441,1,441,Deletion,"BCAM, EX3-4DEL",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2007-03-01,True,"[""BCAM""]","[""RCV000000470""]","[""PubMed:17319831""]",1
1,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003876,1,3876,Insertion,"HEXB, 18-BP INS",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,1990-10-15,True,"[""HEXB""]","[""RCV000004080""]","[""PubMed:2170400"", ""PubMed:868875""]",1
2,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000004793,1,4793,Deletion,"PRX, 1-BP DEL, 247C",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2002-06-01,True,"[""PRX""]","[""RCV000005060""]","[""PubMed:12112076""]",1
3,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000005178,1,5178,single nucleotide variant,"ADCY10, 923C-T",2010-12-01,2019-03-29,...,None,None,risk factor,None,2002-04-01,False,"[""ADCY10""]","[""RCV000005486""]","[""PubMed:11932268""]",1
4,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000005179,1,5179,single nucleotide variant,"ADCY10, 1438+30T-C",2010-12-01,2019-03-29,...,None,None,risk factor,None,2002-04-01,False,"[""ADCY10""]","[""RCV000005487""]","[""PubMed:11932268""]",1



 2021-12 RCV legacy rcv_state


,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000470,4,VCV000000441,1,441,None,None,...,None,Pathogenic,no assertion criteria provided,2007-03-01,True,"[""BLOOD GROUP--LUTHERAN NULL""]","[""OMIM:612773.0003"", ""OMIM:612773.0004"", ""OMIM...",[],"[""PubMed:17319831""]",1
1,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000004080,2,VCV000003876,1,3876,None,None,...,None,Pathogenic,no assertion criteria provided,1990-10-15,True,"[""Hexosaminidase B (paris)""]","[""MedGen:C4016989""]",[],"[""PubMed:2170400"", ""PubMed:868875""]",1
2,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005060,3,VCV000004793,1,4793,None,None,...,None,Pathogenic,no assertion criteria provided,2002-06-01,True,"[""Autosomal recessive Dejerine-Sottas syndrome""]","[""MedGen:CN069172""]",[],"[""PubMed:12112076""]",1
3,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005486,4,VCV000005178,1,5178,None,None,...,None,risk factor,no assertion criteria provided,2002-04-01,False,"[""Hypercalciuria, absorptive, susceptibility to""]",[],[],"[""PubMed:11932268""]",1
4,2021-12,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005487,4,VCV000005179,1,5179,None,None,...,None,risk factor,no assertion criteria provided,2002-04-01,False,"[""Hypercalciuria, absorptive, susceptibility to""]",[],[],"[""PubMed:11932268""]",1


## 27. What to do after this run

### If `batch4_go = true`

Production Batch 4 is accepted:

**2021-10 → 2021-12 ✅**

Record:
- `batch_elapsed_hours`
- individual source runtimes
- detected accelerator
- host logical CPU count
- concurrency wall-clock ratio

### Next Stage-01 production interval

If Batch 4 passes, continue with:

**Batch 5: 2022-01 → 2022-03**

Keep the same frozen Stage-00 manifest, RCV + VCV normalized-table contract, full-stream NCBI MD5 verification, maximum concurrency of 2, resumable completion markers, and memory-bounded QC.



### Scientific boundary

Still no instability outcome, survival model, or GES-R forecast.

**archive → normalize → link → define transitions → leakage audit → dynamic features → forecasting model**
